# SearchSense
## 02 — Feature Engineering

**Goal:** Prepare raw Criteo data for model training by handling missing values, 
encoding categorical features, and engineering new signals.  
**Author:** Ramya Manasa Amancherla

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle
import os

# Load raw data
col_names = ["click"] + [f"num_{i}" for i in range(13)] + [f"cat_{i}" for i in range(26)]
df = pd.read_csv("data/dac/train.txt", sep="\t", header=None, names=col_names, nrows=500_000)

print(f"Loaded {len(df):,} rows")
print(f"Shape: {df.shape}")

Loaded 500,000 rows
Shape: (500000, 40)


## 1. Drop High-Missing Features
num_11 is missing in 76.8% of rows. Imputing this would introduce more noise than signal, so we drop it entirely.

In [3]:
# Drop num_11 due to 76.8% missing rate
df = df.drop(columns=['num_11'])

# Separate feature types
num_cols = [f'num_{i}' for i in range(13) if i != 11]
cat_cols = [f'cat_{i}' for i in range(26)]

print(f"Numerical features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Shape after dropping num_11: {df.shape}")

Numerical features: 12
Categorical features: 26
Shape after dropping num_11: (500000, 39)


## 2. Handle Missing Values
Numerical features are median imputed. Categorical features are filled with a constant unknown token.

In [4]:
from sklearn.impute import SimpleImputer

# Median imputation for numerical features
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

# Fill missing categoricals with 'unknown'
df[cat_cols] = df[cat_cols].fillna('unknown')

# Verify no missing values remain
total_missing = df[num_cols + cat_cols].isnull().sum().sum()
print(f"Total missing values remaining: {total_missing}")
print(f"Numerical features missing: {df[num_cols].isnull().sum().sum()}")
print(f"Categorical features missing: {df[cat_cols].isnull().sum().sum()}")

Total missing values remaining: 0
Numerical features missing: 0
Categorical features missing: 0


## 3. Encode Categorical Features
Categorical features are hashed strings. We use frequency encoding, replacing each category 
with how often it appears in the dataset. This preserves signal without exploding dimensionality.

In [5]:
# Frequency encoding for categorical features
# Replaces each category value with its frequency in the dataset
freq_encoders = {}

for col in cat_cols:
    freq_map = df[col].value_counts(normalize=True).to_dict()
    df[col] = df[col].map(freq_map)
    freq_encoders[col] = freq_map

print("Frequency encoding complete.")
print(f"Sample encoded values for cat_0: {df['cat_0'].head().values}")

# Save encoders for use during inference
os.makedirs("backend", exist_ok=True)
with open("backend/freq_encoders.pkl", "wb") as f:
    pickle.dump(freq_encoders, f)

print("Encoders saved to backend/freq_encoders.pkl")

Frequency encoding complete.
Sample encoded values for cat_0: [0.16658  0.16658  0.001618 0.16658  0.049626]
Encoders saved to backend/freq_encoders.pkl


### Key Findings
Three types of new features were created. Ratio and interaction features capture the relationship 
between num_9 and num_10, which were the two strongest predictors identified in EDA. Log transforms 
reduce the impact of extreme outliers in skewed features, which is important because most numerical 
features showed heavy right skew in the distribution analysis. The filled count feature captures 
how much information is available per row, which can itself be a signal of ad quality.

## 4. Feature Engineering
Creating new features that capture additional signal beyond the raw inputs.

In [6]:
# Ratio of high-signal features identified in EDA
df['num9_num10_ratio'] = df['num_9'] / (df['num_10'] + 1e-6)

# Interaction between top positive predictors
df['num9_num0_interaction'] = df['num_9'] * df['num_0']

# Log transform skewed features to reduce outlier impact
for col in ['num_0', 'num_1', 'num_2', 'num_3', 'num_9', 'num_10']:
    df[f'{col}_log'] = np.log1p(df[col])

# Total non-null numerical signal strength
df['num_filled_count'] = (df[num_cols] != 0).sum(axis=1)

print("New features created:")
new_features = ['num9_num10_ratio', 'num9_num0_interaction', 'num_filled_count']
log_features = [f'{c}_log' for c in ['num_0', 'num_1', 'num_2', 'num_3', 'num_9', 'num_10']]
print(new_features + log_features)
print(f"\nShape after feature engineering: {df.shape}")

New features created:
['num9_num10_ratio', 'num9_num0_interaction', 'num_filled_count', 'num_0_log', 'num_1_log', 'num_2_log', 'num_3_log', 'num_9_log', 'num_10_log']

Shape after feature engineering: (500000, 48)


/Users/ramyaamancherla/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/ramyaamancherla/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


### Key Findings
9 new features were added, bringing the total feature count from 40 to 48. Ratio and interaction 
terms capture relationships between the two strongest predictors from EDA. Log transforms on 6 
skewed features will help the model learn more stable patterns by compressing the long tail of 
extreme values.

## 5. Prepare Final Dataset
Split into train and validation sets and save for model training.

In [7]:
from sklearn.model_selection import train_test_split

# Define final feature set
feature_cols = (
    num_cols +
    cat_cols +
    ['num9_num10_ratio', 'num9_num0_interaction', 'num_filled_count'] +
    [f'{c}_log' for c in ['num_0', 'num_1', 'num_2', 'num_3', 'num_9', 'num_10']]
)

X = df[feature_cols]
y = df['click']

# 80/20 train-validation split, stratified to preserve click rate
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:   {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Train click rate: {y_train.mean():.3f}")
print(f"Val click rate:   {y_val.mean():.3f}")

# Save processed data
os.makedirs("data", exist_ok=True)
X_train.to_csv("data/X_train.csv", index=False)
X_val.to_csv("data/X_val.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_val.to_csv("data/y_val.csv", index=False)

# Save feature column names for inference
with open("backend/feature_cols.pkl", "wb") as f:
    pickle.dump(feature_cols, f)

print("\nData saved to data/ folder")
print("Feature columns saved to backend/feature_cols.pkl")

Training set:   (400000, 47)
Validation set: (100000, 47)
Train click rate: 0.256
Val click rate:   0.256

Data saved to data/ folder
Feature columns saved to backend/feature_cols.pkl


### Key Findings
The 500K dataset was split into 400K training rows and 100K validation rows. The click rate is 
identical across both splits at 25.6%, confirming the stratified split worked correctly. The 
model will be evaluated on data it has never seen during training.